<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9a_JANA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercise 9a — exact JANA Figure 3 and topology-matched hybrid CE

This notebook makes the Figure 3 comparison genuinely apples-to-apples.  For both Gaussian Mixture and SIR it first retrains the **exact task-specific JANA model** defined in `bayesflow-org/JANA-Paper` at commit `6cbbc94...`, using the BayesFlow commit pinned by that repository.  It then constructs a second posterior/likelihood pair with the **same topology and training campaign**, freezes both flows, and only then learns our equal-prior multiclass CE correction.

The distinction among the three controls is deliberate:

1. **Exact JANA:** posterior and likelihood NLLs are optimized jointly through `AmortizedPosteriorLikelihood`, exactly as in the paper repository.
2. **Matched separate flows:** the same two networks are trained on the same 10,000 simulations with the same epochs, batches, optimizer, cosine schedule, clipping, validation budget, transforms, and latent bases, but with separate `Trainer` instances.
3. **Hybrid CE:** the matched separate flows are frozen and corrected by ten plain `4×1024` ReLU classifiers trained only with equal-prior three-class cross entropy.

The published 1,000-dataset Figure 3 test banks and 250 posterior draws are retained.  Thus changes in the SBC curves can be attributed to the optimizer partition and CE correction rather than to an unrelated PyTorch flow architecture.


In [ ]:
# Campaign controls
PROFILE = "PAPER"       # PAPER | SMOKE
SEED = 1
FORCE_LEGACY_FLOWS = False
LOAD_CLASSIFIERS_IF_AVAILABLE = True
FORCE_SELECTED_CONTEXT_PROPOSALS = False

if PROFILE == "PAPER":
    SIMULATION_BUDGET = 10_000
    POSTERIOR_DRAWS = 250
    POSTERIOR_CANDIDATES = 4_096
    LIKELIHOOD_CANDIDATES = 1_024
    CLASSIFIER_MEMBERS = 10
    CLASSIFIER_EPOCHS = 250
    CLASSIFIER_WIDTH = 1_024
    CLASSIFIER_LAYERS = 4
else:
    SIMULATION_BUDGET = 512
    POSTERIOR_DRAWS = 32
    POSTERIOR_CANDIDATES = 128
    LIKELIHOOD_CANDIDATES = 64
    CLASSIFIER_MEMBERS = 1
    CLASSIFIER_EPOCHS = 2
    CLASSIFIER_WIDTH = 96
    CLASSIFIER_LAYERS = 2

if PROFILE == "PAPER":
    required = {
        "simulation budget": (SIMULATION_BUDGET, 10_000),
        "posterior draws": (POSTERIOR_DRAWS, 250),
        "classifier members": (CLASSIFIER_MEMBERS, 10),
        "classifier epochs": (CLASSIFIER_EPOCHS, 250),
        "classifier width": (CLASSIFIER_WIDTH, 1_024),
        "classifier layers": (CLASSIFIER_LAYERS, 4),
    }
    mismatches = {key: pair for key, pair in required.items() if pair[0] != pair[1]}
    if mismatches:
        raise ValueError(f"The PAPER profile contract was modified: {mismatches}")


In [ ]:
# Repository, persistent artifacts, pinned JANA source, and isolated legacy environment.
import hashlib
import json
import math
import os
import random
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
JANA_REPOSITORY = "https://github.com/bayesflow-org/JANA-Paper.git"
JANA_COMMIT = "6cbbc94faf0aa85147986f7f9516d13a52551bd4"
BAYESFLOW_COMMIT = "153dfefadd347717b7aeb9c4872a4b51ac04e83c"
DRIVER_SCHEMA = "jana_fig3_exact_task_specific_v1"
TASKS = ("gaussian_mixture", "sir")
TASK_TITLES = {"gaussian_mixture": "Gaussian Mixture", "sir": "SIR"}
IN_COLAB = "google.colab" in sys.modules

def run(*args, env=None):
    command = [str(arg) for arg in args]
    print("+", " ".join(command))
    subprocess.run(command, check=True, env=env)

if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    REPO_DIR = Path("/content/nsbi-lhc-toolkit")
    if not (REPO_DIR / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, REPO_DIR, env=clone_env,
        )
    else:
        run("git", "-C", REPO_DIR, "fetch", "origin", BRANCH)
        run("git", "-C", REPO_DIR, "checkout", BRANCH)
        run("git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH)
    run(
        "git", "-C", REPO_DIR, "sparse-checkout", "set",
        "src", "workshops/ml4hep_tifr_colab",
    )
    TUTORIAL_DIR = REPO_DIR / "workshops" / "ml4hep_tifr_colab"
    ARTIFACT_ROOT = Path(
        f"/content/drive/MyDrive/hybrid_nsbi_ml/exercise_9a_JANA_fig3_exact/seed_{SEED:02d}"
    )
    LEGACY_ENV = Path("/content/jana-fig3-py311")
    JANA_SOURCE = Path("/content/JANA-Paper-fig3")
else:
    candidates = [Path.cwd(), Path.cwd() / "workshops" / "ml4hep_tifr_colab"]
    TUTORIAL_DIR = next(
        path for path in candidates if (path / "jana_fig3_paper_driver.py").is_file()
    )
    ARTIFACT_ROOT = Path.cwd() / "exercise_9a_JANA_fig3_artifacts" / f"seed_{SEED:02d}"
    LEGACY_ENV = Path("/tmp/jana-fig3-py311")
    source_candidates = [Path.cwd() / "jana-upstream", Path.cwd() / "JANA-Paper-fig3"]
    JANA_SOURCE = next((path for path in source_candidates if (path / ".git").is_dir()), source_candidates[-1])

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVER = TUTORIAL_DIR / "jana_fig3_paper_driver.py"
REQUIREMENTS = TUTORIAL_DIR / "requirements_jana_fig3_paper.txt"
for required in (DRIVER, REQUIREMENTS):
    if not required.is_file():
        raise FileNotFoundError(required)

if not (JANA_SOURCE / ".git").is_dir():
    run("git", "clone", "--filter=blob:none", "--no-checkout", JANA_REPOSITORY, JANA_SOURCE)
run("git", "-C", JANA_SOURCE, "fetch", "--depth", "1", "origin", JANA_COMMIT)
run("git", "-C", JANA_SOURCE, "checkout", "--detach", JANA_COMMIT)
jana_head = subprocess.check_output(
    ["git", "-C", str(JANA_SOURCE), "rev-parse", "HEAD"], text=True
).strip()
if jana_head != JANA_COMMIT:
    raise RuntimeError(f"Pinned JANA checkout mismatch: {jana_head}")

if shutil.which("uv") is None:
    run(sys.executable, "-m", "pip", "install", "-q", "uv")
UV = shutil.which("uv") or str(Path(sys.executable).parent / "uv")
UV_ENV = os.environ.copy()
UV_ENV.setdefault("UV_CACHE_DIR", str(LEGACY_ENV.parent / "jana-fig3-uv-cache"))
UV_ENV.setdefault("UV_PYTHON_INSTALL_DIR", str(LEGACY_ENV.parent / "jana-fig3-uv-python"))
LEGACY_PYTHON = LEGACY_ENV / "bin" / "python"
if not LEGACY_PYTHON.is_file():
    run(UV, "venv", "--python", "3.11", LEGACY_ENV, env=UV_ENV)

requirements_digest = hashlib.sha256(REQUIREMENTS.read_bytes()).hexdigest()
stamp = LEGACY_ENV / ".jana_requirements_sha256"
if not stamp.is_file() or stamp.read_text().strip() != requirements_digest:
    run(
        UV, "pip", "install", "--python", LEGACY_PYTHON,
        "--requirement", REQUIREMENTS, env=UV_ENV,
    )
    # JAX is unused here.  The 2023 file pins jax but not jaxlib, so a modern
    # resolver can create an incompatible pair which prevents TensorFlow import.
    run(UV, "pip", "uninstall", "--python", LEGACY_PYTHON, "jax", "jaxlib", env=UV_ENV)
    stamp.write_text(requirements_digest)

if str(TUTORIAL_DIR) not in sys.path:
    sys.path.insert(0, str(TUTORIAL_DIR))
print("Tutorial directory:", TUTORIAL_DIR)
print("Persistent artifact root:", ARTIFACT_ROOT)
print("Pinned JANA source:", JANA_SOURCE)
print("Legacy JANA Python:", LEGACY_PYTHON)


## 1. Exact Figure 3 architecture and optimization contract

These settings are read directly from `experiments/benchmarks/benchmark_7_gaussian_mixture.ipynb` and `benchmark_9_sir.ipynb` at the pinned JANA commit.  “Same topology” therefore means the task-specific network below, not the generic Exercise 9 RQS flow previously used in this notebook.

| Task | Posterior $q_\phi(\theta\mid x)$ | Likelihood $q_\eta(x\mid\theta)$ | Latent bases | Paper training |
|---|---|---|---|---|
| Gaussian Mixture | 6 spline couplings; learnable permutation; one 64-unit `swish` layer; L2 $10^{-4}$; dropout 0.05 | 6 spline couplings; fixed permutation; same coupling network | unit Gaussian | 10,000 simulations; 150 epochs; batch 64; 300 validation simulations |
| SIR | 6 spline couplings; fixed permutation; one 64-unit ReLU layer; L2 $10^{-4}$; dropout 0.05 | 8 affine couplings; fixed permutation; same coupling network | Student-$t_{50}$ in dimensions 2 and 10 | 10,000 simulations; 250 epochs; batch 32; 300 validation simulations |

The default BayesFlow optimizer is Adam at $5\times10^{-4}$ with its cosine decay and global gradient clipping at 1.  Gaussian Mixture retains BayesFlow's $x/12$ scaling followed by JANA's $\theta/10$ neural coordinate.  SIR retains the mock summary $x[:,:,0]$ and the independent Gaussian stabilization noise with standard deviation $10^{-3}$.  The jointly trained JANA pair and the separately trained hybrid proposal pair use the same fixed simulator bank.


In [ ]:
# Train or reuse the exact legacy flows, one task at a time.
for task_index, task in enumerate(TASKS):
    command = [
        str(LEGACY_PYTHON), str(DRIVER),
        "--mode", "train",
        "--task", task,
        "--artifact-root", str(ARTIFACT_ROOT),
        "--jana-root", str(JANA_SOURCE),
        "--simulation-budget", str(SIMULATION_BUDGET),
        "--seed", str(SEED + 10_000 * task_index),
        "--posterior-candidates", str(POSTERIOR_CANDIDATES),
        "--likelihood-candidates", str(LIKELIHOOD_CANDIDATES),
        "--posterior-draws", str(POSTERIOR_DRAWS),
    ]
    if PROFILE == "SMOKE":
        command.append("--smoke")
    if FORCE_LEGACY_FLOWS:
        command.append("--force")
    run(*command)


In [ ]:
# Load and validate immutable outputs from the pinned environment.
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

LEGACY = {}
METADATA = {}
for task_index, task in enumerate(TASKS):
    task_root = ARTIFACT_ROOT / task
    metadata_path = task_root / "legacy_flow_metadata.json"
    arrays_path = task_root / "legacy_flow_outputs.npz"
    metadata = json.loads(metadata_path.read_text())
    arrays = dict(np.load(arrays_path, allow_pickle=False))
    expected = {
        "driver_schema": DRIVER_SCHEMA,
        "task": task,
        "seed": SEED + 10_000 * task_index,
        "simulation_budget": SIMULATION_BUDGET,
        "posterior_candidates": POSTERIOR_CANDIDATES,
        "likelihood_candidates": LIKELIHOOD_CANDIDATES,
        "posterior_draws": POSTERIOR_DRAWS,
        "smoke": PROFILE == "SMOKE",
        "jana_paper_commit": JANA_COMMIT,
        "bayesflow_commit": BAYESFLOW_COMMIT,
        "test_datasets": 1_000,
        "classifier_schema": "S_P_L_equal_prior_jana_model_coordinates_v1",
    }
    mismatches = {
        key: (metadata.get(key), value)
        for key, value in expected.items() if metadata.get(key) != value
    }
    if mismatches:
        raise RuntimeError(f"Incompatible {task} legacy artifacts: {mismatches}")
    for name, values in arrays.items():
        if not np.isfinite(values).all():
            raise FloatingPointError(f"Non-finite {task} array {name}")
    LEGACY[task] = arrays
    METADATA[task] = metadata
    display(pd.DataFrame([
        {"task": task, "array": name, "shape": str(values.shape), "dtype": str(values.dtype)}
        for name, values in arrays.items()
    ]))


## 2. Freeze both density routes, then learn only the CE correction

For model-coordinate pairs $y=(\theta,x)$, define three equal-prior class laws

\[
S(y)=p(\theta,x),\qquad
P(y)=p(x)q_\phi(\theta\mid x),\qquad
L(y)=p(\theta)q_\eta(x\mid\theta).
\]

After the separate JANA-topology flows are frozen, a three-class classifier estimates

\[
r_P(\theta,x)=\frac{S}{P}=\frac{p(\theta\mid x)}{q_\phi(\theta\mid x)},
\qquad
r_L(\theta,x)=\frac{S}{L}=\frac{p(x\mid\theta)}{q_\eta(x\mid\theta)}.
\]

The corrected conditionals are $q_\phi r_P$ and $q_\eta r_L$.  No normalization or bridge residual is placed in the training loss.  The classifier is exactly the CE-only Exercise 9b contract: ten independent plain `Linear–ReLU` MLPs, four hidden layers of width 1024, no dropout, weight decay, normalization, residual path, or output bound; 250 epochs; and learning rate $10^{-4}\rightarrow10^{-9}$ in factors of ten every 40 epochs.  Inference uses the arithmetic mean of member-wise positive softmax probability quotients.


In [ ]:
# Build grouped S/P/L rows and train or load the CE-only classifier ensembles.
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from utils_exercise9b_contract import (
    INITIAL_LEARNING_RATE,
    LEARNING_RATE_DROP_FACTOR,
    LEARNING_RATE_STEP_EPOCHS,
    MINIMUM_LEARNING_RATE,
)

if (
    INITIAL_LEARNING_RATE != 1e-4
    or MINIMUM_LEARNING_RATE != 1e-9
    or LEARNING_RATE_DROP_FACTOR != 0.1
    or LEARNING_RATE_STEP_EPOCHS != 40
):
    raise RuntimeError("The shared Exercise 9b CE schedule changed; bump this campaign explicitly.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_S, CLASS_P, CLASS_L = 0, 1, 2
CLASSIFIER_ROW_BATCH_BUDGET = 32

def seed_everything(seed):
    random.seed(int(seed))
    np.random.seed(int(seed) % (2**32))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

def array_digest(*arrays):
    digest = hashlib.sha256()
    for array in arrays:
        value = np.ascontiguousarray(np.asarray(array))
        digest.update(str(value.shape).encode())
        digest.update(str(value.dtype).encode())
        digest.update(value.view(np.uint8))
    return digest.hexdigest()

class PlainThreeClassMLP(nn.Module):
    def __init__(self, input_dim, width, hidden_layers):
        super().__init__()
        layers = []
        current = int(input_dim)
        for _ in range(int(hidden_layers)):
            layers.extend([nn.Linear(current, int(width)), nn.ReLU()])
            current = int(width)
        layers.append(nn.Linear(current, 3))
        self.network = nn.Sequential(*layers)

    def forward(self, values):
        return self.network(values)

def scheduled_learning_rate(epoch):
    return max(
        MINIMUM_LEARNING_RATE,
        INITIAL_LEARNING_RATE
        * LEARNING_RATE_DROP_FACTOR ** (int(epoch) // LEARNING_RATE_STEP_EPOCHS),
    )

def set_learning_rate(optimizer, epoch):
    value = scheduled_learning_rate(epoch)
    for group in optimizer.param_groups:
        group["lr"] = value
    return value

def class_ce(model, group_batch):
    group_batch = group_batch.to(device)
    logits = model(group_batch.reshape(-1, group_batch.shape[-1]))
    labels = torch.arange(3, device=device).repeat(len(group_batch))
    return F.cross_entropy(logits, labels)

@torch.no_grad()
def validation_ce(model, values, chunk_size=512):
    model.eval()
    total = 0.0
    for start in range(0, len(values), int(chunk_size)):
        batch = values[start:start + int(chunk_size)]
        total += float(class_ce(model, batch).cpu()) * len(batch)
    return total / len(values)

CLASSIFIERS = {}
CLASSIFIER_TRANSFORMS = {}
CLASSIFIER_HISTORIES = {}
CLASSIFIER_CONFIGS = {}

for task_index, task in enumerate(TASKS):
    legacy = LEGACY[task]
    theta = legacy["theta_train"].astype(np.float32)
    x = legacy["x_train"].astype(np.float32)
    groups_raw = np.stack([
        np.column_stack([theta, x]),
        np.column_stack([legacy["theta_p"].astype(np.float32), x]),
        np.column_stack([theta, legacy["x_l"].astype(np.float32)]),
    ], axis=1).astype(np.float32)
    if groups_raw.shape != (SIMULATION_BUDGET, 3, theta.shape[1] + x.shape[1]):
        raise RuntimeError(f"Unexpected {task} classifier groups {groups_raw.shape}")

    split_rng = np.random.default_rng(SEED + 50_000 + 10_000 * task_index)
    order = split_rng.permutation(len(groups_raw))
    n_train = int(0.8 * len(order))
    train_index, validation_index = order[:n_train], order[n_train:]
    if np.intersect1d(train_index, validation_index).size:
        raise RuntimeError(f"{task}: classifier split leakage")
    train_raw = groups_raw[train_index]
    validation_raw = groups_raw[validation_index]
    flat_train = train_raw.reshape(-1, train_raw.shape[-1])
    center = flat_train.mean(axis=0, dtype=np.float64).astype(np.float32)
    scale = flat_train.std(axis=0, dtype=np.float64).astype(np.float32)
    scale = np.where(scale > 1e-6, scale, 1.0).astype(np.float32)
    transform = lambda values, c=center, s=scale: (
        (np.asarray(values, dtype=np.float32) - c) / s
    ).astype(np.float32)
    train_groups = transform(train_raw)
    validation_groups = transform(validation_raw)
    train_tensor = torch.as_tensor(train_groups, dtype=torch.float32)
    validation_tensor = torch.as_tensor(validation_groups, dtype=torch.float32)

    metadata_digest = hashlib.sha256(
        (ARTIFACT_ROOT / task / "legacy_flow_metadata.json").read_bytes()
    ).hexdigest()
    config = {
        "task": task,
        "input_dim": int(groups_raw.shape[-1]),
        "width": CLASSIFIER_WIDTH,
        "hidden_layers": CLASSIFIER_LAYERS,
        "members": CLASSIFIER_MEMBERS,
        "epochs": CLASSIFIER_EPOCHS,
        "batch_size_row_budget": CLASSIFIER_ROW_BATCH_BUDGET,
        "initial_learning_rate": INITIAL_LEARNING_RATE,
        "minimum_learning_rate": MINIMUM_LEARNING_RATE,
        "learning_rate_drop_factor": LEARNING_RATE_DROP_FACTOR,
        "learning_rate_step_epochs": LEARNING_RATE_STEP_EPOCHS,
        "objective": "equal_prior_multiclass_ce_only",
        "input_transform": "fixed_training_column_mean_std_v1",
        "regularization": "none",
        "seed": SEED + 10_000 * task_index,
        "flow_artifact_metadata": metadata_digest,
        "groups_sha256": array_digest(groups_raw, train_index, validation_index),
    }
    checkpoint_dir = ARTIFACT_ROOT / task / "plain_three_class_classifier"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    def checkpoint_path(member, root=checkpoint_dir):
        return root / f"member_{member:02d}.pt"

    loaded = []
    if LOAD_CLASSIFIERS_IF_AVAILABLE and all(
        checkpoint_path(member).is_file() for member in range(CLASSIFIER_MEMBERS)
    ):
        for member in range(CLASSIFIER_MEMBERS):
            try:
                saved = torch.load(checkpoint_path(member), map_location=device, weights_only=False)
            except TypeError:
                saved = torch.load(checkpoint_path(member), map_location=device)
            if saved.get("config") != {**config, "member": member}:
                raise RuntimeError(f"Incompatible classifier checkpoint {checkpoint_path(member)}")
            if not np.array_equal(np.asarray(saved["center"]), center):
                raise RuntimeError(f"{task}: classifier center mismatch")
            if not np.array_equal(np.asarray(saved["scale"]), scale):
                raise RuntimeError(f"{task}: classifier scale mismatch")
            model = PlainThreeClassMLP(
                config["input_dim"], config["width"], config["hidden_layers"]
            ).to(device)
            model.load_state_dict(saved["state_dict"], strict=True)
            model.eval()
            loaded.append({"model": model, "history": saved["history"]})
        print(task, ": loaded", len(loaded), "frozen CE members")
    else:
        for member in range(CLASSIFIER_MEMBERS):
            member_seed = SEED + 100_003 + 10_000 * task_index + 10_007 * member
            seed_everything(member_seed)
            model = PlainThreeClassMLP(
                config["input_dim"], config["width"], config["hidden_layers"]
            ).to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=INITIAL_LEARNING_RATE)
            loader = DataLoader(
                TensorDataset(train_tensor),
                batch_size=max(1, CLASSIFIER_ROW_BATCH_BUDGET // 3),
                shuffle=True,
                generator=torch.Generator().manual_seed(member_seed + 1),
            )
            history = {"train_ce": [], "validation_ce": [], "learning_rate": []}
            best_state, best_value = None, math.inf
            for epoch in range(CLASSIFIER_EPOCHS):
                learning_rate = set_learning_rate(optimizer, epoch)
                model.train()
                running, rows = 0.0, 0
                for (group_batch,) in loader:
                    optimizer.zero_grad(set_to_none=True)
                    loss = class_ce(model, group_batch)
                    if not bool(torch.isfinite(loss)):
                        raise FloatingPointError(f"{task}: non-finite classifier CE")
                    loss.backward()
                    optimizer.step()
                    running += float(loss.detach().cpu()) * len(group_batch)
                    rows += len(group_batch)
                heldout = validation_ce(model, validation_tensor)
                history["train_ce"].append(running / rows)
                history["validation_ce"].append(heldout)
                history["learning_rate"].append(learning_rate)
                if heldout < best_value - 1e-6:
                    best_value = heldout
                    best_state = copy.deepcopy(model.state_dict())
                if epoch == 0 or (epoch + 1) % 40 == 0 or epoch + 1 == CLASSIFIER_EPOCHS:
                    print(
                        f"{task} member {member + 1}/{CLASSIFIER_MEMBERS}, "
                        f"epoch {epoch + 1:03d}: CE={history['train_ce'][-1]:.5f}, "
                        f"val={heldout:.5f}, lr={learning_rate:.1e}"
                    )
            if best_state is None:
                raise RuntimeError(f"{task}: no finite classifier checkpoint")
            model.load_state_dict(best_state)
            model.eval()
            torch.save({
                "state_dict": best_state,
                "config": {**config, "member": member},
                "center": center,
                "scale": scale,
                "history": history,
            }, checkpoint_path(member))
            loaded.append({"model": model, "history": history})

    CLASSIFIERS[task] = loaded
    CLASSIFIER_TRANSFORMS[task] = {"center": center, "scale": scale}
    CLASSIFIER_HISTORIES[task] = [pack["history"] for pack in loaded]
    CLASSIFIER_CONFIGS[task] = config
    print(task, "classifier parameters/member:", sum(p.numel() for p in loaded[0]["model"].parameters()))

print("Classifier device:", device)


## 3. Correct the two routes in Figure 3 coordinates

The first Figure 3 column is ordinary posterior SBC: $x\sim p(x\mid\theta)$ and posterior samples are drawn at that simulator output.  The second is JANA's joint test: $\widetilde x\sim q_\eta(x\mid\theta)$ and the posterior is then evaluated at the surrogate output.

For Hybrid CE, the first column importance-resamples the frozen posterior proposals with $r_P$.  For the second column it first importance-resamples the frozen likelihood proposals with $r_L$, passes the selected surrogate output back through the same frozen posterior flow, and then importance-resamples those posterior proposals with $r_P$.  The second legacy call below only samples from the already-frozen posterior checkpoint; it performs no optimization.


In [ ]:
# Frozen CE inference, normalization checks, posterior correction, and corrected surrogate x.
@torch.no_grad()
def predict_ratios(task, points, batch_size=16_384):
    points = np.asarray(points, dtype=np.float32)
    center = CLASSIFIER_TRANSFORMS[task]["center"]
    scale_values = CLASSIFIER_TRANSFORMS[task]["scale"]
    posterior_chunks, likelihood_chunks = [], []
    tiny = torch.finfo(torch.float64).tiny
    for start in range(0, len(points), int(batch_size)):
        standardized = ((points[start:start + int(batch_size)] - center) / scale_values).astype(np.float32)
        tensor = torch.as_tensor(standardized, device=device)
        posterior_mean = None
        likelihood_mean = None
        for pack in CLASSIFIERS[task]:
            probabilities = torch.softmax(pack["model"](tensor).to(torch.float64), dim=1)
            r_p = probabilities[:, CLASS_S] / probabilities[:, CLASS_P].clamp_min(tiny)
            r_l = probabilities[:, CLASS_S] / probabilities[:, CLASS_L].clamp_min(tiny)
            member_scale = float(len(CLASSIFIERS[task]))
            posterior_mean = r_p / member_scale if posterior_mean is None else posterior_mean + r_p / member_scale
            likelihood_mean = r_l / member_scale if likelihood_mean is None else likelihood_mean + r_l / member_scale
        posterior_chunks.append(posterior_mean.cpu().numpy())
        likelihood_chunks.append(likelihood_mean.cpu().numpy())
    result = np.column_stack([np.concatenate(posterior_chunks), np.concatenate(likelihood_chunks)])
    if not np.isfinite(result).all() or np.any(result <= 0.0):
        raise FloatingPointError(f"{task}: invalid deployed CE probability quotient")
    return result

def normalized_positive(values):
    values = np.asarray(values, dtype=np.float64)
    if not np.isfinite(values).all() or np.any(values <= 0.0):
        raise FloatingPointError("Importance weights must be finite and positive")
    maximum = float(values.max())
    scaled = values / maximum
    return scaled / scaled.sum(dtype=np.float64)

def corrected_posterior_samples(task, candidates, contexts, n_draws, seed, context_batch_size=8):
    candidates = np.asarray(candidates, dtype=np.float32)
    contexts = np.asarray(contexts, dtype=np.float32)
    rng = np.random.default_rng(int(seed))
    samples, ess_values, mass_values = [], [], []
    for start in range(0, len(contexts), int(context_batch_size)):
        candidate_batch = candidates[start:start + int(context_batch_size)]
        context_batch = contexts[start:start + int(context_batch_size)]
        repeated_context = np.repeat(
            context_batch[:, None, :], candidate_batch.shape[1], axis=1
        )
        points = np.concatenate([candidate_batch, repeated_context], axis=-1)
        weights_batch = predict_ratios(
            task, points.reshape(-1, points.shape[-1])
        )[:, 0].reshape(candidate_batch.shape[:2])
        for candidate, weights in zip(candidate_batch, weights_batch):
            probabilities = normalized_positive(weights)
            selected = rng.choice(
                len(candidate), size=int(n_draws), replace=True, p=probabilities
            )
            samples.append(candidate[selected])
            ess_values.append(1.0 / np.sum(probabilities**2))
            mass_values.append(float(np.mean(weights)))
    return np.stack(samples), np.asarray(ess_values), np.asarray(mass_values)

def corrected_likelihood_contexts(task, candidates, theta, seed, context_batch_size=16):
    candidates = np.asarray(candidates, dtype=np.float32)
    theta = np.asarray(theta, dtype=np.float32)
    rng = np.random.default_rng(int(seed))
    selected_contexts, ess_values, mass_values = [], [], []
    for start in range(0, len(theta), int(context_batch_size)):
        candidate_batch = candidates[start:start + int(context_batch_size)]
        theta_batch = theta[start:start + int(context_batch_size)]
        repeated_theta = np.repeat(
            theta_batch[:, None, :], candidate_batch.shape[1], axis=1
        )
        points = np.concatenate([repeated_theta, candidate_batch], axis=-1)
        weights_batch = predict_ratios(
            task, points.reshape(-1, points.shape[-1])
        )[:, 1].reshape(candidate_batch.shape[:2])
        for candidate, weights in zip(candidate_batch, weights_batch):
            probabilities = normalized_positive(weights)
            selected_contexts.append(
                candidate[rng.choice(len(candidate), p=probabilities)]
            )
            ess_values.append(1.0 / np.sum(probabilities**2))
            mass_values.append(float(np.mean(weights)))
    return np.stack(selected_contexts), np.asarray(ess_values), np.asarray(mass_values)

STAGE_ONE = {}
NORMALIZATION_ROWS = []
for task_index, task in enumerate(TASKS):
    legacy = LEGACY[task]
    hybrid_true, posterior_ess, posterior_mass = corrected_posterior_samples(
        task, legacy["posterior_true_candidates"], legacy["x_test"],
        POSTERIOR_DRAWS, SEED + 200_000 + 10_000 * task_index,
    )
    hybrid_contexts, likelihood_ess, likelihood_mass = corrected_likelihood_contexts(
        task, legacy["likelihood_candidates"], legacy["theta_test"],
        SEED + 210_000 + 10_000 * task_index,
    )
    selected_path = ARTIFACT_ROOT / task / "selected_hybrid_contexts.npz"
    np.savez_compressed(selected_path, contexts=hybrid_contexts, theta=legacy["theta_test"])
    STAGE_ONE[task] = {
        "hybrid_true": hybrid_true,
        "hybrid_contexts": hybrid_contexts,
        "posterior_ess": posterior_ess,
        "likelihood_ess": likelihood_ess,
    }
    NORMALIZATION_ROWS.extend([
        {
            "task": task, "route": "posterior", "mean_E_q_ratio": float(posterior_mass.mean()),
            "RMS_log_E_q_ratio": float(np.sqrt(np.mean(np.log(posterior_mass) ** 2))),
            "median_candidate_ESS": float(np.median(posterior_ess)),
            "candidate_count": int(POSTERIOR_CANDIDATES),
        },
        {
            "task": task, "route": "likelihood", "mean_E_q_ratio": float(likelihood_mass.mean()),
            "RMS_log_E_q_ratio": float(np.sqrt(np.mean(np.log(likelihood_mass) ** 2))),
            "median_candidate_ESS": float(np.median(likelihood_ess)),
            "candidate_count": int(LIKELIHOOD_CANDIDATES),
        },
    ])

normalization_table = pd.DataFrame(NORMALIZATION_ROWS)
display(normalization_table)
normalization_table.to_csv(ARTIFACT_ROOT / "post_training_normalization_and_ess.csv", index=False)


In [ ]:
# Ask the frozen legacy posterior flow for proposals at the CE-corrected surrogate contexts.
for task_index, task in enumerate(TASKS):
    command = [
        str(LEGACY_PYTHON), str(DRIVER),
        "--mode", "selected-contexts",
        "--task", task,
        "--artifact-root", str(ARTIFACT_ROOT),
        "--jana-root", str(JANA_SOURCE),
        "--simulation-budget", str(SIMULATION_BUDGET),
        "--seed", str(SEED + 10_000 * task_index),
        "--posterior-candidates", str(POSTERIOR_CANDIDATES),
        "--likelihood-candidates", str(LIKELIHOOD_CANDIDATES),
        "--posterior-draws", str(POSTERIOR_DRAWS),
    ]
    if FORCE_SELECTED_CONTEXT_PROPOSALS:
        command.append("--force")
    run(*command)


In [ ]:
# Exact JANA fractional ranks, matched-separate control, Hybrid CE, and metrics.
def fractional_ranks(samples, truth):
    samples = np.asarray(samples)
    truth = np.asarray(truth)
    return np.sum(samples < truth[:, None, :], axis=1) / samples.shape[1]

def rank_metrics(ranks, task, method, diagnostic, parameter_names):
    grid = np.linspace(0.0, 1.0, 201)
    rows = []
    for index, parameter in enumerate(parameter_names):
        empirical = (ranks[:, index, None] <= grid[None, :]).mean(axis=0)
        residual = empirical - grid
        rows.append({
            "task": task,
            "method": method,
            "diagnostic": diagnostic,
            "parameter": parameter,
            "MAE_ECDF": float(np.mean(np.abs(residual))),
            "RMSE_ECDF": float(np.sqrt(np.mean(residual**2))),
            "KS": float(np.max(np.abs(residual))),
            "mean_fractional_rank": float(np.mean(ranks[:, index])),
            "n_calibration": int(len(ranks)),
            "posterior_draws": int(POSTERIOR_DRAWS),
        })
    return rows

RESULTS = {}
METRIC_ROWS = []
for task_index, task in enumerate(TASKS):
    legacy = LEGACY[task]
    selected = dict(np.load(
        ARTIFACT_ROOT / task / "selected_context_posterior_candidates.npz",
        allow_pickle=False,
    ))
    if not np.array_equal(selected["contexts"], STAGE_ONE[task]["hybrid_contexts"]):
        raise RuntimeError(f"{task}: selected-context proposal mismatch")
    hybrid_surrogate, joint_posterior_ess, joint_posterior_mass = corrected_posterior_samples(
        task, selected["posterior_candidates"], selected["contexts"],
        POSTERIOR_DRAWS, SEED + 220_000 + 10_000 * task_index,
    )
    methods = {
        "Exact JANA": {
            "simulator": fractional_ranks(legacy["joint_posterior_true"], legacy["theta_test"]),
            "surrogate": fractional_ranks(legacy["joint_posterior_surrogate"], legacy["theta_test"]),
        },
        "Matched separate": {
            "simulator": fractional_ranks(legacy["separate_posterior_true"], legacy["theta_test"]),
            "surrogate": fractional_ranks(legacy["separate_posterior_surrogate"], legacy["theta_test"]),
        },
        "Hybrid CE": {
            "simulator": fractional_ranks(STAGE_ONE[task]["hybrid_true"], legacy["theta_test"]),
            "surrogate": fractional_ranks(hybrid_surrogate, legacy["theta_test"]),
        },
    }
    parameter_names = METADATA[task]["topology"].get(
        "parameter_names", [r"$\theta_1$", r"$\theta_2$"]
    )
    if task == "sir":
        parameter_names = [r"$\beta$", r"$\gamma$"]
    for method, diagnostics in methods.items():
        for diagnostic, ranks in diagnostics.items():
            METRIC_ROWS.extend(rank_metrics(
                ranks, task, method,
                "posterior SBC" if diagnostic == "simulator" else "joint surrogate SBC",
                parameter_names,
            ))
    RESULTS[task] = {
        "methods": methods,
        "hybrid_surrogate": hybrid_surrogate,
        "joint_posterior_ess": joint_posterior_ess,
        "joint_posterior_mass": joint_posterior_mass,
        "parameter_names": parameter_names,
    }
    np.savez_compressed(
        ARTIFACT_ROOT / task / "figure3_rank_results.npz",
        exact_jana_simulator=methods["Exact JANA"]["simulator"],
        exact_jana_surrogate=methods["Exact JANA"]["surrogate"],
        matched_separate_simulator=methods["Matched separate"]["simulator"],
        matched_separate_surrogate=methods["Matched separate"]["surrogate"],
        hybrid_ce_simulator=methods["Hybrid CE"]["simulator"],
        hybrid_ce_surrogate=methods["Hybrid CE"]["surrogate"],
        posterior_ess=STAGE_ONE[task]["posterior_ess"],
        likelihood_ess=STAGE_ONE[task]["likelihood_ess"],
        joint_posterior_ess=joint_posterior_ess,
        parameter_names=np.asarray(parameter_names),
    )

metric_table = pd.DataFrame(METRIC_ROWS)
display(metric_table)
metric_table.to_csv(ARTIFACT_ROOT / "figure3_exact_jana_vs_hybrid_metrics.csv", index=False)


## 4. Figure 3 head-to-head

The plot retains JANA's fractional-rank definition, $R=M^{-1}\sum_m\mathbf 1[\theta^{(m)}<\theta_{\rm true}]$, rather than adding randomized jitter.  The gray region is the same simultaneous uniform-ECDF construction used by the pinned BayesFlow diagnostics.  Line style identifies the estimator; color identifies the parameter.


In [ ]:
# Figure 3-style simultaneous ECDF-difference comparison.
import matplotlib.pyplot as plt
from scipy import stats

def get_coverage_probs(z, uniforms, chunk_size=64):
    n_samples = uniforms.shape[1]
    minimum = np.ones(uniforms.shape[0], dtype=np.float64)
    # Algebraically identical to BayesFlow's vectorized implementation, but
    # chunking the evaluation grid avoids a temporary M x K x N boolean cube.
    for start in range(0, len(z), int(chunk_size)):
        z_chunk = z[start:start + int(chunk_size)]
        empirical = np.sum(
            z_chunk[None, :, None] >= uniforms[:, None, :], axis=-1
        ) / n_samples
        lower_tail = stats.binom(n_samples, z_chunk).cdf(n_samples * empirical)
        upper_tail = stats.binom(n_samples, z_chunk).cdf(n_samples * empirical - 1)
        minimum = np.minimum(
            minimum, np.minimum(lower_tail, 1 - upper_tail).min(axis=1)
        )
    return 2 * minimum

def simultaneous_ecdf_bands(num_samples, confidence=0.95, seed=0):
    rng = np.random.default_rng(int(seed))
    z = np.linspace(1e-5, 1 - 1e-5, min(int(num_samples), 1000))
    uniforms = rng.uniform(size=(1000, int(num_samples)))
    gamma = np.percentile(get_coverage_probs(z, uniforms), 100 * (1 - confidence))
    lower = stats.binom(num_samples, z).ppf(gamma / 2) / num_samples
    upper = stats.binom(num_samples, z).ppf(1 - gamma / 2) / num_samples
    return z, lower - z, upper - z

figure, axes = plt.subplots(2, 2, figsize=(12.2, 8.2), sharex=True, sharey=True, constrained_layout=True)
colors = ["#0b3c8c", "#b22222"]
styles = {
    "Exact JANA": (":", 2.0),
    "Matched separate": ("--", 1.5),
    "Hybrid CE": ("-", 2.2),
}
for row, task in enumerate(TASKS):
    parameter_names = RESULTS[task]["parameter_names"]
    for column, (diagnostic, title) in enumerate([
        ("simulator", "Posterior SBC: simulator output"),
        ("surrogate", "Joint SBC: surrogate output"),
    ]):
        axis = axes[row, column]
        z, lower, upper = simultaneous_ecdf_bands(
            len(LEGACY[task]["theta_test"]), seed=SEED + 300_000 + row
        )
        axis.fill_between(z, lower, upper, color="0.7", alpha=0.30, label="95% simultaneous band")
        for parameter_index, parameter_name in enumerate(parameter_names):
            for method, (linestyle, linewidth) in styles.items():
                ranks = RESULTS[task]["methods"][method][diagnostic][:, parameter_index]
                x_rank = np.sort(ranks)
                y_ecdf = np.arange(1, len(x_rank) + 1) / len(x_rank)
                axis.plot(
                    x_rank, y_ecdf - x_rank,
                    color=colors[parameter_index], linestyle=linestyle, linewidth=linewidth,
                    label=f"{parameter_name}, {method}",
                )
        axis.axhline(0.0, color="black", linewidth=0.8)
        axis.set_title(f"{TASK_TITLES[task]} — {title}")
        axis.grid(alpha=0.25)
        if row == 1:
            axis.set_xlabel("fractional rank statistic")
        if column == 0:
            axis.set_ylabel("ECDF difference")
        axis.legend(fontsize=7.2, ncol=2, loc="lower center")

figure_path = ARTIFACT_ROOT / "figure3_exact_jana_vs_topology_matched_hybrid_ce.png"
figure.savefig(figure_path, dpi=220, bbox_inches="tight")
figure.savefig(figure_path.with_suffix(".pdf"), bbox_inches="tight")
plt.show()
print("Saved:", figure_path)


In [ ]:
# Legacy flow histories and the ten CE histories.
figure, axes = plt.subplots(2, 2, figsize=(12, 7.5), constrained_layout=True)
for row, task in enumerate(TASKS):
    flow_paths = {
        "joint JANA": ARTIFACT_ROOT / task / "joint_jana_history",
        "separate posterior": ARTIFACT_ROOT / task / "separate_posterior_history",
        "separate likelihood": ARTIFACT_ROOT / task / "separate_likelihood_history",
    }
    for label, stem in flow_paths.items():
        csv_path, json_path = stem.with_suffix(".csv"), stem.with_suffix(".json")
        if csv_path.is_file():
            history = pd.read_csv(csv_path)
            loss_columns = [
                column for column in history.columns
                if "loss" in column.lower() and "val" not in column.lower()
            ]
            for column in loss_columns:
                axes[row, 0].plot(
                    history[column].to_numpy(), alpha=0.75,
                    label=f"{label}: {column}",
                )
        elif json_path.is_file():
            train_losses = np.asarray(
                json.loads(json_path.read_text())["train_losses"], dtype=np.float64
            )
            n_density_losses = 2 if label == "joint JANA" else 1
            density_names = ("posterior NLL", "likelihood NLL") if n_density_losses == 2 else ("NLL",)
            for column, density_name in enumerate(density_names):
                axes[row, 0].plot(
                    train_losses[:, column], alpha=0.75,
                    label=f"{label}: {density_name}",
                )
        else:
            raise FileNotFoundError(f"No history artifact for {stem}")
    for member, history in enumerate(CLASSIFIER_HISTORIES[task]):
        axes[row, 1].plot(history["validation_ce"], alpha=0.45, label=f"member {member + 1}")
    axes[row, 0].set(title=f"{TASK_TITLES[task]} — legacy BayesFlow NLL", xlabel="stored step", ylabel="loss")
    axes[row, 1].set(title=f"{TASK_TITLES[task]} — frozen-checkpoint selection", xlabel="epoch", ylabel="held-out CE")
    axes[row, 0].legend(fontsize=6.5)
    if CLASSIFIER_MEMBERS <= 4:
        axes[row, 1].legend(fontsize=7)
    for axis in axes[row]:
        axis.grid(alpha=0.2)

history_path = ARTIFACT_ROOT / "figure3_training_histories.png"
figure.savefig(history_path, dpi=180, bbox_inches="tight")
plt.show()


## 5. Reading the result

- **Exact JANA versus matched separate** isolates the fact that the two NLLs are optimized through one joint amortizer versus two independent trainers.  Their analytic objective is additive, but initialization, validation generation, stochastic preprocessing, and optimizer trajectories can still produce finite-sample differences.
- **Matched separate versus Hybrid CE** isolates the effect of the CE correction because the flow proposals are literally the same frozen checkpoints.
- A correction can improve posterior SBC yet degrade joint surrogate SBC.  That means $q_\phi r_P$ improved while the corrected composition $q_\eta r_L\rightarrow q_\phi r_P$ did not; it should not be summarized as an overall improvement.
- Candidate ESS and $E_q[r]$ are reported for both routes.  Low ESS is a failure of the finite proposal calculation, not evidence that the corrected density is intrinsically poor.
- This is an exact-algorithm retraining at a fixed reproducible seed, not a claim that the random weights equal the historical checkpoint checked into JANA-Paper.  The published test banks are used exactly.  A paper-level superiority statement still requires multiple independent 10,000-simulation campaigns and uncertainty across those campaigns.


In [ ]:
# Final immutable run manifest.
manifest = {
    "status": "complete",
    "exercise": "Exercise_9a_JANA",
    "profile": PROFILE,
    "seed": SEED,
    "tasks": list(TASKS),
    "simulation_budget": SIMULATION_BUDGET,
    "posterior_draws": POSTERIOR_DRAWS,
    "jana_paper_commit": JANA_COMMIT,
    "bayesflow_commit": BAYESFLOW_COMMIT,
    "driver_schema": DRIVER_SCHEMA,
    "baseline": {
        "label": "exact JANA Figure 3 algorithm, paired retraining",
        "joint_objective": "AmortizedPosteriorLikelihood with the repository's task-specific networks",
        "historical_checkpoint_weights": False,
        "published_test_banks": True,
    },
    "legacy_metadata": METADATA,
    "hybrid_proposals": {
        "training": "separate Trainer instances after reconstructing the exact task-specific JANA flows",
        "only_intentional_flow_difference": "optimizer partition",
    },
    "classifier": {
        "configs": CLASSIFIER_CONFIGS,
        "objective": "equal-prior multiclass CE only",
        "ensemble_rule": "arithmetic mean of member-wise softmax probability quotients",
        "diagnostics_in_gradients_or_selection": False,
    },
    "normalization_and_ess": normalization_table.to_dict(orient="records"),
    "metrics": metric_table.to_dict(orient="records"),
    "figure": str(figure_path),
    "training_histories": str(history_path),
}
manifest_path = ARTIFACT_ROOT / "run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True, default=str))
print("Run complete:", manifest_path)
